In [1]:
!pip install -q "google-cloud-aiplatform>=1.38"

In [2]:
! pip3 install -q --upgrade --user google-cloud-aiplatform
! gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=BscPtKE7srVxBiZYhrjDQ0v0vHuKcB&access_type=offline&code_challenge=u7pJdHRkbSxku7uiE6c552fKNb5WlYcrDiai4_moVGg&code_challenge_method=S256


Credentials saved to file: [/home/rishihazra/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "sat-solving" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [3]:
# Initialize Vertex AI
import vertexai

vertexai.init(project='sat-solving')
from vertexai.preview.generative_models import GenerativeModel
from google.api_core.exceptions import InvalidArgument

import os
import sys
from typing import List, Tuple
import system_messages
from utils.dataset import SatDataset, custom_collate
from utils.data_analysis import log_data_sample, sample_in_context
from torch.utils.data import DataLoader
from dotmap import DotMap
from tqdm import tqdm
import torch
import time

os.environ["ROOT_PATH"] = os.getcwd()
sys.path.append(os.environ["ROOT_PATH"])

In [11]:
# os.system("auth.authenticate_user()")

def query_gemini(model_name: str, preferences: str, in_context_examples: str) -> Tuple[str, int, int]:
    """
    :param model_name: gpt-4, gpt-3.5
    :param preferences: food preferences of people
    :return: raw output generated by gpt-* model, # prompt_tokens, # completion_tokens
    """
    model_name = "gemini-pro"
    generation_model = GenerativeModel(model_name)
    
    PROMPT = f"System prompt: {system_message}\n{in_context_examples}\nPreferences: {preferences}\nSolution: "

    response = generation_model.generate_content(PROMPT)
    
    return (response.text, response._raw_response.usage_metadata.prompt_token_count, response._raw_response.usage_metadata.candidates_token_count)


if __name__ == "__main__":
    ablation = 'menu'  # 'menu', 'sat', 'translate'
    two_sat_flag = False
    few_shot = 3  # 0 for zero_shot
    job_number = ''
    append = '_2sat' if two_sat_flag else ''
    # system message to prompt the model
    # different system messages for different ablations
    system_message = system_messages.names[ablation]
    model_name = 'gemini-pro'  # gpt-4, gpt-3.5, llama-2-70b, llama-2-13b, text-bison@002, gemini-pro
    data_path = os.path.join(os.environ["ROOT_PATH"], f'dataset{append}.pkl')
    sat_dataset = SatDataset(root_path=os.environ["ROOT_PATH"], data_path=data_path)

    batch_size = 1
    if few_shot > 0:
        in_shot_examples = sample_in_context(few_shot, model_name, ablation)
    sat_loader = DataLoader(sat_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate)

    for data_sample in tqdm(sat_loader):
        data_sample = [DotMap(sample) for sample in data_sample]
        data_input = data_sample[0].preferences if ablation in ['menu', 'translate'] \
            else data_sample[0].formula
        try:
            gen_out, num_prompt_tokens, num_completion_tokens = query_gemini(model_name, data_input, in_shot_examples)
            log_data_sample(model_name, data_sample[0].num_vars, data_sample[0].num_clauses,
                            data_sample[0].formula, data_sample[0].is_sat,
                            data_sample[0].preferences, data_sample[0].menu_items,
                            gen_out, num_prompt_tokens, num_completion_tokens,
                            ablation=ablation, job_num=job_number, few_shot=few_shot, two_sat_flag=two_sat_flag)
            # time.sleep(10)
        except InvalidArgument:
            continue
        except ValueError:
            continue

  3%|████▉                                                                                                                                                                        | 680/24000 [2:21:24<80:49:29, 12.48s/it]


InternalServerError: 500 Received RST_STREAM with error code 2